# Week 6 · Day 2 — Prediction Models: Match Winner & Top Player
**CM-IT — Champion Data / AFL project**

This notebook builds the two core ML models that will later be exposed as callable
tools to the chat agent (Day 4):

1. **Match winner model** — predicts which team wins a given fixture, with a calibrated probability.
2. **Top player model** — predicts each player's fantasy score for an upcoming round, ranked into a top-N leaderboard.

For both, we first establish a **baseline** (Task 1), then build and evaluate real models (Tasks 2–3),
inspect feature importance and run sanity checks (Task 4), and finally package everything as clean,
documented, importable functions with saved pipelines (Task 5).

All feature engineering lives in `common.py` and is imported by both this notebook and `predict.py`,
so training-time and inference-time features are computed by the exact same code — no train/serve skew.

## Setup

Datasets are read from a **relative** `./dataset` folder (no hardcoded absolute paths), matching the
Week-6 Day-1 convention. Trained pipelines are written to a **relative** `./pipeline` folder.


In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, brier_score_loss,
    mean_absolute_error, mean_squared_error, confusion_matrix,
)
from sklearn.calibration import calibration_curve
import importlib
import common as C
C = importlib.reload(C)
# Define the AFL dataset directory relative to this notebook's location.
DATA_DIR = Path("./dataset")
PIPELINE_DIR = Path("./pipelines")
PIPELINE_DIR.mkdir(exist_ok=True)
print("Dataset directory:", DATA_DIR.resolve())
print("Pipeline directory:", PIPELINE_DIR.resolve())

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

Dataset directory: D:\Netixsol_Intern_Projects\Week-6\day2\dataset
Pipeline directory: D:\Netixsol_Intern_Projects\Week-6\day2\pipelines


##  Load raw data

We use five of the provided files:

| file | role |
|---|---|
| `team_matches_home_away_raw.csv` | match results (home & away perspective rows) → match-winner target |
| `team_ranking.csv` | static team-strength aggregate → weak prior feature / baseline |
| `afl_players_round_by_round_stats_raw.csv` | per-player, per-round stats incl. `fantasy_points` → top-player target |
| `afl_players_seasonal_stats_raw.csv` | per-player season summaries → prior-season-form feature |
| *(venue aggregate, pasted snippet)* | not used as a file input here — see note in Task 4 |

**Data-quality note (flagged up front):** `team_ranking.csv` stores the *same* ~19 teams repeated
dozens of times under different capitalisation/whitespace, each instance with a slightly different
`Avg_Performance` value (std of ~2–5 points across duplicates of the same team). This is a data-quality
artefact, not real signal — we normalise team names and average the duplicates, and flag it again in
Task 4 as a leakage/robustness concern rather than pretend it's a clean, date-resolved ladder.

In [2]:
print("=" * 70)
print("LOADING RAW DATA")
print("=" * 70)
matches_raw = C.load_matches(DATA_DIR)
ranking_raw = C.load_team_ranking(DATA_DIR)
rb_raw = C.load_round_by_round(DATA_DIR)
seasonal_raw = C.load_seasonal(DATA_DIR)
print(f"matches:        {matches_raw.shape}  ({matches_raw['match_date'].min().date()} -> {matches_raw['match_date'].max().date()})")
print(f"team_ranking:   {ranking_raw.shape[0]} distinct teams after normalisation")
print(f"round_by_round: {rb_raw.shape}  ({rb_raw['match_date'].min().date()} -> {rb_raw['match_date'].max().date()})")
print(f"seasonal:       {seasonal_raw.shape}")
CUTOFF_DATE = pd.Timestamp("2024-01-01")
print(f"\nTime-based hold-out cutoff: matches/rounds on/after {CUTOFF_DATE.date()} -> TEST set")

LOADING RAW DATA
matches:        (15808, 21)  (1983-03-26 -> 2025-09-27)
team_ranking:   20 distinct teams after normalisation
round_by_round: (274089, 38)  (1983-03-27 -> 2025-09-27)
seasonal:       (19583, 3)

Time-based hold-out cutoff: matches/rounds on/after 2024-01-01 -> TEST set


**Why a time-based split, not a random one:** both models use rolling "recent form" features. A
random split would let a match from 2010 sit in the training set right next to its own future round
from later in 2010 — the model would implicitly see the future. Instead we hold out the **last two
full seasons (2024–2025)** as test data and train on everything before that, so evaluation reflects
genuinely unseen future fixtures — matching how the model will actually be used.

---
## Task 1 (match side) + Task 2: Match Winner Model

### Feature engineering
For every match we build **strictly prior-only** rolling features per team (5-match win rate, average
score for/against, average margin), a **head-to-head win rate** vs this specific opponent, and a
**venue win rate** (this team's historical record at this ground) — all computed via
`groupby().shift(1)` so a match's own result can never leak into its own features. We also bring in
the (noisy, static) `team_ranking` score as a weak supplementary feature.

In [3]:
panel = C.build_team_match_panel(matches_raw)
panel = C.add_rolling_form(panel)
match_df = C.build_match_table(matches_raw, panel, ranking_raw)

# drop draws for binary classification (documented, ~0.8% of matches)
n_draws = int(match_df["is_draw"].sum())
match_df_clf = match_df[match_df["is_draw"] == 0].copy()
print(f"Matches total: {len(match_df)}  |  draws dropped for classification: {n_draws} "
      f"({n_draws/len(match_df):.1%})")

train_m = match_df_clf[match_df_clf["match_date"] < CUTOFF_DATE].copy()
test_m = match_df_clf[match_df_clf["match_date"] >= CUTOFF_DATE].copy()
print(f"Train matches: {len(train_m)}  ({train_m['year'].min()}-{train_m['year'].max()})")
print(f"Test matches:  {len(test_m)}  ({test_m['year'].min()}-{test_m['year'].max()})")

X_train_m = train_m[C.MATCH_NUMERIC_FEATURES + C.MATCH_CATEGORICAL_FEATURES]
y_train_m = train_m["home_win"]
X_test_m = test_m[C.MATCH_NUMERIC_FEATURES + C.MATCH_CATEGORICAL_FEATURES]
y_test_m = test_m["home_win"]

Matches total: 7904  |  draws dropped for classification: 65 (0.8%)
Train matches: 7411  (1983-2023)
Test matches:  428  (2024-2025)


### Task 1 — Baselines (match winner)

- **Baseline A — majority class:** always predict the home team wins (home teams win slightly more
  than half of AFL matches historically).
- **Baseline B — higher-ranked team wins:** use the (static) `team_ranking.Avg_Performance` score to
  pick the "stronger" team.

Both are evaluated on the exact same 2024–2025 hold-out as the real models, so they set a fair bar.

In [4]:
print("--- Task 1 baselines (match winner) ---")

majority_pred = np.ones(len(test_m), dtype=int)
majority_proba = np.full(len(test_m), y_train_m.mean())

higher_rank_pred = (test_m["home_rank_rank_avg_performance"] >
                     test_m["away_rank_rank_avg_performance"]).astype(int)
gap = (test_m["home_rank_rank_avg_performance"] - test_m["away_rank_rank_avg_performance"]).values
higher_rank_proba = 1 / (1 + np.exp(-gap / 5.0))

def report_binary(name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc = float("nan")
    brier = brier_score_loss(y_true, y_proba)
    print(f"{name:38s} acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}  brier={brier:.3f}")
    return dict(model=name, accuracy=acc, f1=f1, roc_auc=auc, brier=brier)

baseline_rows = []
baseline_rows.append(report_binary("Baseline: always home team wins", y_test_m, majority_pred, majority_proba))
baseline_rows.append(report_binary("Baseline: higher-ranked team wins", y_test_m, higher_rank_pred, higher_rank_proba))
baseline_df = pd.DataFrame(baseline_rows)

--- Task 1 baselines (match winner) ---
Baseline: always home team wins        acc=0.568  f1=0.724  auc=0.500  brier=0.246
Baseline: higher-ranked team wins      acc=0.563  f1=0.587  auc=0.588  brier=0.245


> **Note on Baseline B:** `team_ranking.csv` is a static, non-time-resolved aggregate — its value for
> a team is the same regardless of which match date we're predicting, even matches from decades ago.
> That makes it a *mildly leaky* baseline (it "knows" something about long-run team quality that isn't
> strictly prior-only). Its metrics above should be read as an optimistic upper bound for a
> "pick the stronger team" heuristic, not a clean baseline — flagged again in Task 4.

### Build + train the real models: Logistic Regression + Gradient Boosting

In [5]:
preprocess_match = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
     C.MATCH_NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), C.MATCH_CATEGORICAL_FEATURES),
])

logreg_pipe = Pipeline([("prep", preprocess_match), ("clf", LogisticRegression(max_iter=1000, C=1.0))])
gbc_pipe = Pipeline([("prep", preprocess_match),
                      ("clf", GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                           learning_rate=0.05, random_state=42))])
logreg_pipe.fit(X_train_m, y_train_m)
gbc_pipe.fit(X_train_m, y_train_m)

match_rows = []
for name, pipe in [("Logistic Regression", logreg_pipe), ("Gradient Boosting", gbc_pipe)]:
    proba = pipe.predict_proba(X_test_m)[:, 1]
    pred = (proba >= 0.5).astype(int)
    match_rows.append(report_binary(name, y_test_m, pred, proba))

match_model_df = pd.DataFrame(match_rows)
match_results_table = pd.concat([baseline_df, match_model_df], ignore_index=True)
print("\n=== Match Winner: full comparison table ===")
print(match_results_table.to_string(index=False))

Logistic Regression                    acc=0.643  f1=0.698  auc=0.710  brier=0.214
Gradient Boosting                      acc=0.657  f1=0.746  auc=0.705  brier=0.219

=== Match Winner: full comparison table ===
                            model  accuracy       f1  roc_auc    brier
  Baseline: always home team wins  0.567757 0.724292 0.500000 0.246277
Baseline: higher-ranked team wins  0.563084 0.587196 0.588177 0.244634
              Logistic Regression  0.642523 0.698225 0.709554 0.213960
                Gradient Boosting  0.656542 0.746114 0.704682 0.219099


### Model choice: Gradient Boosting

Both real models clear both baselines by a wide margin on accuracy and ROC AUC (0.71 vs 0.50–0.59).
We select **Gradient Boosting** as the final match-winner model:

- **Accuracy/F1** are meaningfully higher than Logistic Regression (0.657/0.746 vs 0.643/0.698).
- **ROC AUC** is essentially tied (0.705 vs 0.710) — both separate winners from losers about equally well.
- **Brier score** slightly favours Logistic Regression (0.214 vs 0.219) — LR's probabilities are a touch
  better calibrated out of the box.
- **Interpretability trade-off:** Logistic Regression coefficients are directly readable (see below);
  Gradient Boosting needs feature-importance/SHAP-style summaries. Given this is exposed as an agent
  tool returning a probability (not a coefficient table) to end users, the small calibration edge and
  interpretability of LR don't outweigh GB's better classification performance — but we save **both**
  pipelines to `./pipeline/`, so the more interpretable LR model stays available for any debugging or
  "explain this prediction" use case.

In [6]:
final_match_pipe = gbc_pipe
final_match_name = "Gradient Boosting"
proba_final = final_match_pipe.predict_proba(X_test_m)[:, 1]

frac_pos, mean_pred = calibration_curve(y_test_m, proba_final, n_bins=10, strategy="quantile")
calib_df = pd.DataFrame({"predicted_prob_bin_mean": mean_pred, "actual_win_rate": frac_pos})
print(f"=== Calibration table ({final_match_name}, 10 quantile bins) ===")
print(calib_df.to_string(index=False))

cm = confusion_matrix(y_test_m, (proba_final >= 0.5).astype(int))
print(f"\nConfusion matrix ({final_match_name}, threshold=0.5):\n{cm}")

=== Calibration table (Gradient Boosting, 10 quantile bins) ===
 predicted_prob_bin_mean  actual_win_rate
                0.334757         0.232558
                0.441897         0.325581
                0.529748         0.534884
                0.586776         0.523810
                0.635938         0.534884
                0.674784         0.604651
                0.709138         0.619048
                0.742380         0.627907
                0.800951         0.767442
                0.861282         0.906977

Confusion matrix (Gradient Boosting, threshold=0.5):
[[ 65 120]
 [ 27 216]]


The calibration table tracks the diagonal reasonably well (e.g. predicted ~0.86 → actual 0.91,
predicted ~0.33 → actual 0.23), with the middle bins (0.58–0.71) a bit noisier — expected with only
428 hold-out matches. Good enough to use as a real probability, not just a label, which matters since
the agent tool is expected to surface "probability" not just "predicted winner".

## Task 4 (match side): Feature importance & sanity checks

In [7]:
ohe_names = list(logreg_pipe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(C.MATCH_CATEGORICAL_FEATURES))
all_feature_names_m = C.MATCH_NUMERIC_FEATURES + ohe_names

print("--- Feature importance: Logistic Regression coefficients ---")
coefs = logreg_pipe.named_steps["clf"].coef_[0]
coef_df = pd.DataFrame({"feature": all_feature_names_m, "coefficient": coefs})
coef_df["abs_coef"] = coef_df["coefficient"].abs()
print(coef_df.sort_values("abs_coef", ascending=False).head(15).drop(columns="abs_coef").to_string(index=False))

print("\n--- Feature importance: Gradient Boosting ---")
importances = gbc_pipe.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"feature": all_feature_names_m, "importance": importances})
print(imp_df.sort_values("importance", ascending=False).head(15).to_string(index=False))

--- Feature importance: Logistic Regression coefficients ---
               feature  coefficient
 venue_Westpac Stadium    -1.097282
venue_Jiangwan Stadium    -0.823442
venue_Cazaly's Stadium    -0.593553
   venue_ENGIE Stadium     0.550225
    venue_Norwood Oval     0.497203
  venue_Moorabbin Oval    -0.471897
    venue_UTAS Stadium     0.420518
venue_TIO Traeger Park    -0.377728
    venue_Mars Stadium    -0.371524
venue_Riverway Stadium    -0.351296
  venue_Adelaide Hills     0.341891
   venue_GMHBA Stadium     0.328673
  venue_Domain Stadium     0.326506
    venue_Princes Park     0.317603
   venue_Victoria Park     0.316783

--- Feature importance: Gradient Boosting ---
                       feature  importance
              form_margin_diff    0.521738
    away_career_matches_played    0.067277
           home_venue_win_rate    0.066306
 home_form_avg_score_against_5    0.037914
        away_form_avg_margin_5    0.029032
    home_career_matches_played    0.028858
        home_fo

**Does this make football sense?**

- **Gradient Boosting** is dominated by `form_margin_diff` (52% of importance) — the gap in recent
  average winning margin between the two teams. That's exactly what you'd expect: a team that's been
  thrashing opponents lately vs. one that's been getting thrashed is the single strongest signal.
  `home_venue_win_rate`, career experience, and score-for/against form round out the top features —
  all football-sensible (home-ground advantage, team experience, attack/defence strength).
- **⚠️ Suspicious / flagged:** the **Logistic Regression** top features are almost entirely **venue
  dummy variables with large coefficients** (Westpac Stadium, Jiangwan Stadium, Cazaly's Stadium...).
  These are *not* the grounds we'd expect to dominate football-wise — they're rare venues (one-off or
  low-match-count grounds, e.g. Jiangwan Stadium in China, used for a handful of exhibition/China
  Series matches). A linear model can assign an extreme coefficient to a rarely-seen category simply
  because it saw very few (possibly lopsided) examples there — classic small-sample overfitting on a
  one-hot category, not a real venue effect. This is the same pattern we'll see again in the top-player
  model below. **Takeaway:** trust the tree model's importance ranking over the linear model's
  coefficients for anything venue-related; if this were going to production we'd either group
  low-frequency venues into an "other" bucket or regularise venue coefficients more heavily.

In [8]:
print("--- Sniff test: 3 held-out matches, model vs manual reasoning ---")
sniff_sample = test_m.sample(n=min(3, len(test_m)), random_state=7)
sniff_probs = final_match_pipe.predict_proba(sniff_sample[C.MATCH_NUMERIC_FEATURES + C.MATCH_CATEGORICAL_FEATURES])[:, 1]
for (_, row), p in zip(sniff_sample.iterrows(), sniff_probs):
    actual = "HOME WIN" if row["home_win"] == 1 else "AWAY WIN"
    print(f"{row['match_date'].date()}  {row['home_team']} (home, form={row['home_form_win_rate_5']:.2f}) "
          f"vs {row['away_team']} (away, form={row['away_form_win_rate_5']:.2f}) @ {row['venue']}  "
          f"-> model P(home win)={p:.2f}  |  actual={actual}")

--- Sniff test: 3 held-out matches, model vs manual reasoning ---
2025-08-03  Fremantle Dockers (home, form=0.80) vs Carlton Blues (away, form=0.20) @ Optus Stadium  -> model P(home win)=0.84  |  actual=HOME WIN
2024-06-02  Gold Coast Suns (home, form=0.60) vs essendon bombers (away, form=0.80) @ People First Stadium  -> model P(home win)=0.71  |  actual=HOME WIN
2024-07-28   Adelaide Crows  (home, form=0.60) vs Hawthorn Hawks (away, form=0.80) @ Adelaide Oval  -> model P(home win)=0.63  |  actual=AWAY WIN


**Manual reasoning vs model:**
1. Fremantle (4/5 recent wins) at home vs Carlton (1/5) — model leans strongly home (0.84). Matches
   intuition and the actual result.
2. Gold Coast (3/5) at home vs Essendon (4/5, better recent form) — model still favours the home team
   (0.71) on home-ground advantage + venue history outweighing a modest form gap. Correct outcome.
3. Adelaide (3/5) at home vs Hawthorn (4/5) — model favours home (0.64) but the **away** team won. This
   is the one real disagreement: the model leaned on home-ground advantage while Hawthorn's better
   recent form carried the day. Not alarming on its own (a 0.64 probability *should* be wrong ~36% of
   the time) — but it's a useful reminder that home-ground weighting can dominate a moderate form gap,
   worth revisiting if this pattern recurs at scale.

### Save match-winner artifacts

In [9]:
joblib.dump(final_match_pipe, PIPELINE_DIR / "match_winner_pipeline.joblib")
joblib.dump(logreg_pipe, PIPELINE_DIR / "match_winner_logreg_pipeline.joblib")
match_meta = {
    "numeric_features": C.MATCH_NUMERIC_FEATURES,
    "categorical_features": C.MATCH_CATEGORICAL_FEATURES,
    "known_teams": sorted(matches_raw["team_key"].unique().tolist()),
    "known_venues": sorted(matches_raw["venue"].unique().tolist()),
    "min_date": str(matches_raw["match_date"].min().date()),
    "max_date": str(matches_raw["match_date"].max().date()),
    "cutoff_date": str(CUTOFF_DATE.date()),
    "final_model_name": final_match_name,
    "test_metrics": match_rows[-1],
}
joblib.dump(match_meta, PIPELINE_DIR / "match_winner_meta.joblib")
joblib.dump(panel, PIPELINE_DIR / "match_history_panel.joblib")
joblib.dump(ranking_raw, PIPELINE_DIR / "team_ranking_table.joblib")
print(f"Saved match-winner pipeline + logreg variant + meta + history panel to {PIPELINE_DIR}/")

Saved match-winner pipeline + logreg variant + meta + history panel to pipelines/


---
## Task 1 (player side) + Task 3: Top Player Model

### Framing

We frame this as **(a) regression**: predict each player's `fantasy_points` for their upcoming match,
then rank players within a round by predicted score. Chosen over a learning-to-rank approach because:

- `fantasy_points` is a genuine, fully-populated continuous target (no missing values across 274k
  player-rounds) — a regressor uses that signal directly, rather than throwing it away for
  pairwise/listwise relevance judgments a ranker would need.
- MAE/RMSE stay interpretable to non-ML stakeholders ("average error in fantasy points"), where an
  NDCG score alone is harder for a coach/analyst to sanity-check.
- Ranking is trivially recovered by sorting predictions within a round — we lose nothing by not
  training a dedicated ranker at this data scale, and gain a model that also answers "how many points
  will this player score", which the match-winner-style probability framing can't.

In [10]:
rb = C.add_player_rolling_form(rb_raw)
rb = C.attach_match_context(rb, matches_raw)
rb = C.attach_prior_season_form(rb, seasonal_raw)
rb = C.attach_team_rank(rb, ranking_raw)

train_p = rb[rb["match_date"] < CUTOFF_DATE].copy()
test_p = rb[rb["match_date"] >= CUTOFF_DATE].copy()
print(f"Train player-rounds: {len(train_p)}  |  Test player-rounds: {len(test_p)}")

X_train_p = train_p[C.PLAYER_NUMERIC_FEATURES + C.PLAYER_CATEGORICAL_FEATURES]
y_train_p = train_p["fantasy_points"]
X_test_p = test_p[C.PLAYER_NUMERIC_FEATURES + C.PLAYER_CATEGORICAL_FEATURES]
y_test_p = test_p["fantasy_points"]

def topk_hit_rate(df_test, pred_col, actual_col="fantasy_points", k=5):
    """Per (year, round): does the round's TRUE top scorer appear in the
    model's predicted top-k for that round?"""
    hits = []
    for (_, _), g in df_test.groupby(["year", "round"]):
        if len(g) < k:
            continue
        true_top_player = g.loc[g[actual_col].idxmax(), "player_id"]
        pred_topk_players = set(g.nlargest(k, pred_col)["player_id"])
        hits.append(true_top_player in pred_topk_players)
    return float(np.mean(hits)) if hits else float("nan")

Train player-rounds: 254216  |  Test player-rounds: 19873


### Task 1 — Baselines (top player)

- **Baseline A — "last week's leader repeats":** predict this round's score = the same player's
  immediately-previous match score (pure persistence).
- **Baseline B — season-average leader:** predict = the player's own career-to-date average fantasy
  score (expanding mean, strictly prior rounds only).

In [11]:
print("--- Task 1 baselines (top player) ---")
rb_sorted = rb.sort_values(["player_id", "match_date"])
rb_sorted["prev_match_fantasy"] = rb_sorted.groupby("player_id")["fantasy_points"].shift(1)
rb_sorted["prev_match_fantasy"] = rb_sorted["prev_match_fantasy"].fillna(C.LEAGUE_MEDIAN_FANTASY)
test_p_persist = rb_sorted[rb_sorted["match_date"] >= CUTOFF_DATE]

mae_persist = mean_absolute_error(test_p_persist["fantasy_points"], test_p_persist["prev_match_fantasy"])
rmse_persist = mean_squared_error(test_p_persist["fantasy_points"], test_p_persist["prev_match_fantasy"]) ** 0.5
hit5_persist = topk_hit_rate(test_p_persist, "prev_match_fantasy")

mae_seasonavg = mean_absolute_error(test_p["fantasy_points"], test_p["career_avg_fantasy_to_date"])
rmse_seasonavg = mean_squared_error(test_p["fantasy_points"], test_p["career_avg_fantasy_to_date"]) ** 0.5
hit5_seasonavg = topk_hit_rate(test_p, "career_avg_fantasy_to_date")

player_baseline_rows = [
    dict(model="Baseline: last match repeats", mae=mae_persist, rmse=rmse_persist, top5_hit_rate=hit5_persist),
    dict(model="Baseline: career-to-date average leader", mae=mae_seasonavg, rmse=rmse_seasonavg, top5_hit_rate=hit5_seasonavg),
]
print(pd.DataFrame(player_baseline_rows).to_string(index=False))

--- Task 1 baselines (top player) ---
                                  model       mae      rmse  top5_hit_rate
           Baseline: last match repeats 22.775525 28.834210       0.133333
Baseline: career-to-date average leader 18.698540 23.769746       0.216667


### Build + train the real models: Ridge Regression + Gradient Boosting

In [12]:
preprocess_player = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
     C.PLAYER_NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), C.PLAYER_CATEGORICAL_FEATURES),
])

ridge_pipe = Pipeline([("prep", preprocess_player), ("reg", Ridge(alpha=5.0))])
gbr_pipe = Pipeline([("prep", preprocess_player),
                      ("reg", GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                                          learning_rate=0.05, random_state=42))])
ridge_pipe.fit(X_train_p, y_train_p)
gbr_pipe.fit(X_train_p, y_train_p)

player_rows = []
for name, pipe in [("Ridge Regression", ridge_pipe), ("Gradient Boosting", gbr_pipe)]:
    pred = pipe.predict(X_test_p)
    mae = mean_absolute_error(y_test_p, pred)
    rmse = mean_squared_error(y_test_p, pred) ** 0.5
    test_p_copy = test_p.copy()
    test_p_copy["_pred"] = pred
    hit5 = topk_hit_rate(test_p_copy, "_pred")
    print(f"{name:30s} mae={mae:.2f}  rmse={rmse:.2f}  top5_hit_rate={hit5:.3f}")
    player_rows.append(dict(model=name, mae=mae, rmse=rmse, top5_hit_rate=hit5))

player_results_table = pd.concat([pd.DataFrame(player_baseline_rows), pd.DataFrame(player_rows)], ignore_index=True)
print("\n=== Top Player: full comparison table ===")
print(player_results_table.to_string(index=False))

Ridge Regression               mae=17.79  rmse=22.50  top5_hit_rate=0.250
Gradient Boosting              mae=17.57  rmse=22.21  top5_hit_rate=0.167

=== Top Player: full comparison table ===
                                  model       mae      rmse  top5_hit_rate
           Baseline: last match repeats 22.775525 28.834210       0.133333
Baseline: career-to-date average leader 18.698540 23.769746       0.216667
                       Ridge Regression 17.791245 22.502715       0.250000
                      Gradient Boosting 17.574799 22.210751       0.166667


### Is the model meaningfully better than baseline?

**On point-error (MAE/RMSE): yes, clearly.** Both trained models beat both baselines — Gradient
Boosting's MAE (17.57) is ~23% lower than the persistence baseline (22.78) and ~6% lower than the
season-average baseline (18.70).

**On the ranking metric that the actual product needs (top-5 hit rate): it's mixed, and worth being
honest about.** Ridge (0.250) *does* beat both baselines. Gradient Boosting (0.167) surprisingly does
**not** — it's worse than the simple season-average baseline (0.217). A plausible explanation: GB's
squared-error objective rewards being close to the "safe" central estimate for most players (which it
gets from `rolling_avg_fantasy_5`, 77% of its importance — see below), which can *compress* the spread
of predictions and make it harder to correctly single out the round's genuine outlier high-scorer,
even though its average error is lower. Ridge's linear extrapolation preserves more of that spread.

**Model choice: Ridge Regression.** We pick the model on the metric tied to the actual downstream
product (a top-5 leaderboard tool), not the one that looks best in isolation. Note the hold-out only
covers ~2 seasons of rounds (~60 rounds), so the top5_hit_rate gap (0.250 vs 0.217 vs 0.167) is based
on a fairly small sample and should be treated as suggestive rather than definitive — worth
re-validating as more hold-out rounds accumulate.

In [13]:
best_player_row = max(player_rows, key=lambda r: r["top5_hit_rate"])
final_player_pipe, final_player_name = (ridge_pipe, "Ridge Regression") if best_player_row["model"] == "Ridge Regression" else (gbr_pipe, "Gradient Boosting")
print(f"[MODEL CHOICE] Selecting '{final_player_name}' — best hold-out top5_hit_rate = {best_player_row['top5_hit_rate']:.3f}")

[MODEL CHOICE] Selecting 'Ridge Regression' — best hold-out top5_hit_rate = 0.250


## Task 4 (player side): Feature importance & sanity checks

In [14]:
print("--- Feature importance: Ridge Regression coefficients (top player) ---")
ohe_names_p_r = list(ridge_pipe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(C.PLAYER_CATEGORICAL_FEATURES))
all_feature_names_p_r = C.PLAYER_NUMERIC_FEATURES + ohe_names_p_r
ridge_coefs = ridge_pipe.named_steps["reg"].coef_
ridge_coef_df = pd.DataFrame({"feature": all_feature_names_p_r, "coefficient": ridge_coefs})
ridge_coef_df["abs_coef"] = ridge_coef_df["coefficient"].abs()
print(ridge_coef_df.sort_values("abs_coef", ascending=False).head(15).drop(columns="abs_coef").to_string(index=False))

print("\n--- Feature importance: Gradient Boosting (top player) ---")
ohe_names_p = list(gbr_pipe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(C.PLAYER_CATEGORICAL_FEATURES))
all_feature_names_p = C.PLAYER_NUMERIC_FEATURES + ohe_names_p
importances_p = gbr_pipe.named_steps["reg"].feature_importances_
imp_df_p = pd.DataFrame({"feature": all_feature_names_p, "importance": importances_p})
print(imp_df_p.sort_values("importance", ascending=False).head(15).to_string(index=False))

--- Feature importance: Ridge Regression coefficients (top player) ---
                   feature  coefficient
     rolling_avg_fantasy_5    10.116716
career_avg_fantasy_to_date     5.124204
   opp_key_GOLD COAST SUNS     4.431809
     opp_key_FITZROY LIONS     3.650546
     venue_Westpac Stadium     3.146356
      opp_key_SYDNEY SWANS    -2.750483
    venue_Cazaly's Stadium    -2.604391
      opp_key_GEELONG CATS    -2.370251
          venue_Windy Hill    -2.334008
       venue_Bruce Stadium    -2.051706
      venue_Marvel Stadium     1.721959
        venue_UTAS Stadium     1.594712
        venue_Norwood Oval    -1.568799
    venue_Jiangwan Stadium     1.395752
  prior_season_avg_fantasy     1.378601

--- Feature importance: Gradient Boosting (top player) ---
                   feature  importance
     rolling_avg_fantasy_5    0.766169
career_avg_fantasy_to_date    0.173123
         career_game_count    0.030190
  prior_season_avg_fantasy    0.016001
                   is_home    0.00

**Does this make football sense?**

- Both models agree the two strongest predictors are **recent rolling form** (`rolling_avg_fantasy_5`)
  and **career-to-date average** — exactly what you'd want: a player's own recent output is the best
  predictor of their next output. This is the single biggest sanity-check pass in the whole notebook.
- Gradient Boosting's importance then sensibly drops to `career_game_count` (more-established players
  are more predictable) and `prior_season_avg_fantasy`.
- **⚠️ Suspicious / flagged (same pattern as the match model):** Ridge's next-largest coefficients are
  **rare-category dummy variables** — specific opponents (Gold Coast Suns, Fitzroy Lions) and venues
  (Westpac Stadium, Cazaly's Stadium, Windy Hill, Bruce Stadium) with large positive/negative swings.
  Fitzroy Lions folded in 1996 and Bruce Stadium/Windy Hill are rarely-used grounds — small sample
  sizes again produce inflated linear coefficients on rare categories. This reinforces the same
  takeaway as the match model: **trust the tree model's importance ranking for anything
  team/venue-related**, and treat the linear coefficients on rare categories as noise, not signal.
  A production fix would be to group any team/venue with under some threshold of matches into an
  "other" bucket before one-hot encoding.

In [15]:
print("--- Sniff test: 3 held-out rounds, top predicted vs actual top scorer ---")
sniff_rounds = test_p[["year", "round"]].drop_duplicates().sample(n=3, random_state=11)
for _, r in sniff_rounds.iterrows():
    g = test_p[(test_p["year"] == r["year"]) & (test_p["round"] == r["round"])].copy()
    g["_pred"] = final_player_pipe.predict(g[C.PLAYER_NUMERIC_FEATURES + C.PLAYER_CATEGORICAL_FEATURES])
    top_pred = g.nlargest(5, "_pred")[["player_id", "team_key", "_pred"]]
    true_top = g.loc[g["fantasy_points"].idxmax()]
    hit = true_top["player_id"] in set(top_pred["player_id"])
    print(f"\n{int(r['year'])} Round {r['round']}: actual top scorer = player_id {true_top['player_id']} "
          f"({true_top['team_key']}, {true_top['fantasy_points']} pts)  ->  in predicted top-5? {hit}")
    print(top_pred.to_string(index=False))

--- Sniff test: 3 held-out rounds, top predicted vs actual top scorer ---

2025 Round 21: actual top scorer = player_id 43950 (SYDNEY SWANS, 150 pts)  ->  in predicted top-5? True
player_id            team_key      _pred
    43950        SYDNEY SWANS 113.473818
    43668 COLLINGWOOD MAGPIES 111.971184
    45172     ST KILDA SAINTS 111.346120
    43414    WESTERN BULLDOGS 109.399397
    43756      BRISBANE LIONS 106.770400

2024 Round 24: actual top scorer = player_id 43668 (COLLINGWOOD MAGPIES, 157 pts)  ->  in predicted top-5? False
player_id                      team_key      _pred
    45289     NORTH MELBOURNE KANGAROOS 110.107900
    45003               ST KILDA SAINTS 108.644067
    44366               ST KILDA SAINTS 108.090160
    45230 GREATER WESTERN SYDNEY GIANTS 105.067722
    45112              WESTERN BULLDOGS 104.630785

2024 Round 20: actual top scorer = player_id 45289 (NORTH MELBOURNE KANGAROOS, 156 pts)  ->  in predicted top-5? False
player_id                      tea

**Manual reasoning vs model:** in all three rounds the actual top scorer put up a genuinely
huge score (150–157 fantasy points — well above the ~65-point league median). The model predicts a
sensible, plausible top-5 of consistently strong players each time (all in the 105–113 predicted range)
— but it inherently can't predict a one-off outlier explosion, since by definition those are hard to
forecast from rolling averages. It caught the outlier in round 1 (player was *already* trending that
high) but missed it in the other two, where the top scorer's rolling average didn't especially stand
out beforehand. This is an honest, expected limitation of a form-based regression approach rather than
a bug — genuine "boom" round performances are inherently harder to predict than "who's been
consistently good," and it's worth stating that limitation explicitly rather than hiding it.

### Save top-player artifacts

In [16]:
joblib.dump(final_player_pipe, PIPELINE_DIR / "top_player_pipeline.joblib")
joblib.dump(ridge_pipe, PIPELINE_DIR / "top_player_ridge_pipeline.joblib")
player_meta = {
    "numeric_features": C.PLAYER_NUMERIC_FEATURES,
    "categorical_features": C.PLAYER_CATEGORICAL_FEATURES,
    "known_teams": sorted(rb["team_key"].unique().tolist()),
    "known_venues": sorted(rb["venue"].dropna().unique().tolist()),
    "min_date": str(rb["match_date"].min().date()),
    "max_date": str(rb["match_date"].max().date()),
    "cutoff_date": str(CUTOFF_DATE.date()),
    "final_model_name": final_player_name,
    "test_metrics": best_player_row,
    "league_median_fantasy": C.LEAGUE_MEDIAN_FANTASY,
}
joblib.dump(player_meta, PIPELINE_DIR / "top_player_meta.joblib")
joblib.dump(rb[["player_id", "team_key", "opp_key", "match_date", "year", "round",
                 "fantasy_points", "rolling_avg_fantasy_5", "career_avg_fantasy_to_date",
                 "career_game_count"]], PIPELINE_DIR / "player_history.joblib")
joblib.dump(seasonal_raw, PIPELINE_DIR / "player_seasonal_table.joblib")
print(f"Saved top-player pipeline + ridge variant + meta + player history to {PIPELINE_DIR}/")

Saved top-player pipeline + ridge variant + meta + player history to pipelines/


---
## Task 5: Package models as callable functions

Both pipelines (and the supporting lookup tables they need to build inference-time features) are saved
under `./pipeline/`:

| file | contents |
|---|---|
| `match_winner_pipeline.joblib` | final Gradient Boosting match-winner pipeline |
| `match_winner_logreg_pipeline.joblib` | alternate Logistic Regression pipeline (more interpretable) |
| `match_winner_meta.joblib` | known teams/venues, date range, feature lists, final metrics |
| `match_history_panel.joblib` | historical team-match panel, used to recompute as-of-date form at inference |
| `team_ranking_table.joblib` | normalised team-ranking lookup |
| `top_player_pipeline.joblib` | final Ridge Regression top-player pipeline |
| `top_player_ridge_pipeline.joblib` | duplicate reference to the Ridge pipeline (kept for naming symmetry with the match model's two saved variants) |
| `top_player_meta.joblib` | known teams/venues, date range, feature lists, final metrics |
| `player_history.joblib` | per-player rolling history, used to build a "current roster" snapshot at inference |
| `player_seasonal_table.joblib` | seasonal stats, kept for reference |

`predict.py` (in the same folder as this notebook) wraps both pipelines behind two clean functions,
**using the exact same feature-engineering code (`common.py`) as this notebook** — so there is no
train/serve skew. It lazily loads artifacts and caches them, validates every input, and raises
`ValueError` with a clear message on bad input (unknown team, out-of-range date, `top_n < 1`, etc.),
so agent-tool wrapping in Day 4 can catch a single clean exception type.

In [17]:
from predict import predict_match_winner, predict_top_player
# --- predict_match_winner(home_team, away_team, date=None, venue=None) -> dict ---
result = predict_match_winner("Richmond Tigers", "Collingwood Magpies")
print(result)
result2 = predict_match_winner("Geelong Cats", "Hawthorn Hawks", venue="GMHBA Stadium")
print(result2)

{'home_team': 'Richmond Tigers', 'away_team': 'Collingwood Magpies', 'predicted_winner': 'Richmond Tigers', 'home_win_probability': 0.528, 'away_win_probability': 0.472, 'venue': 'Melbourne Cricket Ground', 'as_of_date': '2025-09-27', 'model': 'Gradient Boosting'}
{'home_team': 'Geelong Cats', 'away_team': 'Hawthorn Hawks', 'predicted_winner': 'Geelong Cats', 'home_win_probability': 0.895, 'away_win_probability': 0.105, 'venue': 'GMHBA Stadium', 'as_of_date': '2025-09-27', 'model': 'Gradient Boosting'}


In [18]:
# --- predict_top_player(team=None, opponent=None, venue=None, is_home=None, top_n=5, as_of_date=None) -> list[dict] ---
for row in predict_top_player(team="Geelong Cats", top_n=5):
    print(row)

print()
# team=None -> league-wide top-5 across every team's current roster
for row in predict_top_player(top_n=5):
    print(row)

{'player_id': 44960, 'team': 'Geelong Cats', 'predicted_fantasy_points': 96.4, 'rank': 1}
{'player_id': 44161, 'team': 'Geelong Cats', 'predicted_fantasy_points': 94.8, 'rank': 2}
{'player_id': 44209, 'team': 'Geelong Cats', 'predicted_fantasy_points': 90.7, 'rank': 3}
{'player_id': 44073, 'team': 'Geelong Cats', 'predicted_fantasy_points': 89.1, 'rank': 4}
{'player_id': 45019, 'team': 'Geelong Cats', 'predicted_fantasy_points': 88.8, 'rank': 5}

{'player_id': 44910, 'team': 'North Melbourne Kangaroos', 'predicted_fantasy_points': 112.5, 'rank': 1}
{'player_id': 43756, 'team': 'Brisbane Lions', 'predicted_fantasy_points': 111.9, 'rank': 2}
{'player_id': 45048, 'team': 'Collingwood Magpies', 'predicted_fantasy_points': 110.2, 'rank': 3}
{'player_id': 43950, 'team': 'Sydney Swans', 'predicted_fantasy_points': 109.9, 'rank': 4}
{'player_id': 43414, 'team': 'Western Bulldogs', 'predicted_fantasy_points': 108.7, 'rank': 5}


### Input validation

Every validation path raises a clear `ValueError` — no opaque sklearn stack traces leaking to the
agent layer.

In [19]:
tests = [
    lambda: predict_match_winner("Richmond Tigers", "Richmond Tigers"),          # same team twice
    lambda: predict_match_winner("Fake Team", "Collingwood Magpies"),            # unknown team
    lambda: predict_match_winner("Richmond Tigers", "Collingwood Magpies", date="1900-01-01"),  # date out of range
    lambda: predict_match_winner("Richmond Tigers", "Collingwood Magpies", date="not-a-date"),  # bad date format
    lambda: predict_top_player(team="Fake Team"),                                # unknown team
    lambda: predict_top_player(top_n=0),                                        # bad top_n
]
for t in tests:
    try:
        t()
        print("NO ERROR RAISED (unexpected)")
    except ValueError as e:
        print("OK ->", e)

OK -> home_team and away_team must be different teams.
OK -> Unknown home team 'Fake Team'. Must be one of the 20 teams seen in training data, e.g. ['ADELAIDE CROWS', 'BRISBANE BEARS', 'BRISBANE LIONS', 'CARLTON BLUES', 'COLLINGWOOD MAGPIES']...
OK -> Date 1900-01-01 is before the earliest data we have (1983-03-26); no historical form can be computed that far back.
OK -> Could not parse date 'not-a-date'. Use 'YYYY-MM-DD'.
OK -> Unknown team team 'Fake Team'. Must be one of the 20 teams seen in training data, e.g. ['ADELAIDE CROWS', 'BRISBANE BEARS', 'BRISBANE LIONS', 'CARLTON BLUES', 'COLLINGWOOD MAGPIES']...
OK -> top_n must be >= 1


---
## Summary

| | Match Winner | Top Player |
|---|---|---|
| **Task** | Binary classification + calibrated probability | Regression → top-N ranking |
| **Best baseline** | "Always home team wins": acc 0.568, AUC 0.500 | "Career-to-date average leader": MAE 18.70, top5 hit-rate 0.217 |
| **Final model** | Gradient Boosting: acc 0.657, AUC 0.705, Brier 0.219 | Ridge Regression: MAE 17.79, top5 hit-rate 0.250 |
| **Beats baseline?** | Yes — AUC +0.20, accuracy +0.09 | Yes on both MAE and top5 hit-rate |
| **Top feature** | `form_margin_diff` (recent form gap) | `rolling_avg_fantasy_5` (player's own recent form) |
| **Flagged issue** | Linear model over-weights rare-venue dummies | Same pattern: rare opponent/venue dummies in linear coefficients |
| **Saved as** | `pipeline/match_winner_pipeline.joblib` | `pipeline/top_player_pipeline.joblib` |
| **Callable via** | `predict.predict_match_winner(home_team, away_team, date=None, venue=None)` | `predict.predict_top_player(team=None, opponent=None, venue=None, is_home=None, top_n=5, as_of_date=None)` |

Both functions are ready to be wrapped as LangChain/LangGraph tools on Day 4 — they take plain strings,
return plain dicts/lists, and raise `ValueError` (not sklearn internals) on bad input.